# Nível 1 — Dados e primeira análise com LLM



**Parte A:** carrega e limpa os dados, normaliza valores para BRL, produz agregações
e aplica duas regras 


**Parte B:** submete um cliente sinalizado a uma LLM para obter um parecer estruturado.



## Carregando dados

In [44]:
import json
import pandas as pd

CAMINHO = "../dados/dados_nivel_1.json"

with open(CAMINHO, encoding="utf-8") as f:
    bruto = json.load(f)

taxa_usd_brl = bruto["taxa_cambio_usd_brl"]
df = pd.DataFrame(bruto["operacoes"])

print(f"Taxa de câmbio USD | BRL: {taxa_usd_brl}")
print(f"Operações carregadas: {len(df)}")
df.head()


Taxa de câmbio USD | BRL: 5.4
Operações carregadas: 20


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,


In [45]:
print(" Dimensões (linhas, colunas) ")
print(df.shape)

print("\nTipo de cada coluna ")
print(df.dtypes)

print("\nValores nulos por coluna ")
print(df.isnull().sum())

print("\nLinhas inteiramente duplicadas")
print(df.duplicated().sum())

print("\nIDs repetidos")
print(df["id"].duplicated().sum())

print("\nMoedas presentes ")
print(df["moeda"].value_counts())

print("\nOperações por canal ")
print(df["canal"].value_counts())




 Dimensões (linhas, colunas) 
(20, 9)

Tipo de cada coluna 
id             object
cliente_id     object
data           object
valor           int64
moeda          object
canal          object
tipo           object
contraparte    object
observacao     object
dtype: object

Valores nulos por coluna 
id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64

Linhas inteiramente duplicadas
1

IDs repetidos
1

Moedas presentes 
moeda
BRL    19
USD     1
Name: count, dtype: int64

Operações por canal 
canal
pix        9
ted        5
boleto     3
cartao     2
especie    1
Name: count, dtype: int64


In [46]:
print("\nObservações")
print(df["observacao"].value_counts())


Observações
observacao
                                   18
remessa internacional               1
data nao capturada pelo sistema     1
Name: count, dtype: int64


In [47]:
df[df["id"].duplicated(keep=False)]


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [48]:
df[df["observacao"] != ""]


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
13,OP-0013,CLI-A-4,2026-03-24,12000,USD,ted,transferencia_recebida,Zeta Importacao,remessa internacional
17,OP-0017,CLI-A-5,None,4300,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema


## Diagnóstico da qualidade dos dados



### 1. Coluna data armazenada como texto

- **O quê:** No dataframe o datatype para data está em texto
- **Quantas linhas:** as 20, é um problema de tipagem da coluna inteira, não de registros específicos.
- **Como detectei:** df.dtypes retornou object para a coluna data. Em pandas, object significa texto, uma coluna de datas deveria estar como datetime64
- **Por que atrapalha:** pois é pedido para sinalizar operações em uma mesma data, a "a princípio funciona se for fazer comparação AAAA-MM-DD mas qualquer coisa adicional, como uma filtragem por mês ou algo semelhante iria quebrar



  ### 2. Data faltante 

- **O quê:** temos um cliente com data none
- **Quantas linhas:** 1.
- **Como detectei:** ao usar o isnull vemos que há um valor na coluna data vazio, e a propria observação existente diz que há uma data não capturada pelo sistema
- **Por que atrapalha:** Pois não é possivél fazer agrupamento com data por conta desse cliente, assim ficando sem grupo nenhum 


### 3. Operação em moeda estrangeira

- **O quê:** existência de um único usuário com USD
- **Quantas linhas:** 1.
- **Como detectei:** primeiramente pelo value.count nas variáveis de moeda, assim de cara ja mostrando que existe outro cambio além do REAL, e também pela observação de remessa internacional
- **Por que atrapalha:** Pelo fato de que a coluna valor guarda 19 valores em BRL, e 1 em USD, os numeros não são comparáveis entre si. O que esta em USD tem o valor de 12.000, que equivale a 64.800 reais, sem converter, toda soma, média e mediana sai errada, quebrando as regras propostas



### .4 Linha duplicada

- **O quê:** há a existência de uma linha completa duplicada 
- **Quantas linhas:** 1.
- **Como detectei:** duplicated.sum mostra a quantia de linhas duplicadas existentes no dataframe, exibindo o número que se deve apagar de linhas para normalizar
- **Por que atrapalha:** Pois existir dois clientes iguais em um dataframe atrapalha a média, as contagens, a mediana, e qualquer operação que envolve usar a coluna completa, pois agregaria um valor duplicado


### Verificado e sem problema

- **ID duplicado** : Consultei também se há ID duplicado, pois 1 sabemos que tem pelo fato de existir a linha duplicada, mas se existisse mais um ID duplicado teriamos um problema pelo fato de termos operações diferentes para o mesmo ID, assim causando um conflito, mas não foi o caso, o ID repetido que foi achado, é o mesmo que remete a linha duplicada 

- **observacao vazia em 18 de 20 linhas:** não é defeito. Campo opcional, a
  maioria das operações não tem observação. Vale registrar, porém, que
  isnull().sum() retorna 0 para essa coluna — o pandas não trata "" como
  nulo. Usei value_counts() para enxergar a distribuição real, e foi assim que
  localizei as 2 linhas com observação preenchida, que se revelaram pistas dos
  problemas 2 e 3



## Limpeza dos dados

Cada tratamento corresponde a um problema do diagnóstico

**1. Linha duplicada** — removida com drop_duplicates(). As duas linhas de
OP-0007 são idênticas, então descartar uma não perde informação.
Usei a forma sem subset, que só remove quando a linha inteira coincide: se
houvesse dois ids iguais com conteúdo diferente, eu não teria como saber qual é o
correto, e apagar às cegas seria pior que manter

**2. Tipagem da data** — pd.to_datetime()` com errors="coerce", que converte
valores inválidos em NaT em vez de lançar exceção. Como já existe um nulo na
coluna, o comportamento padrão (errors="raise") interromperia a limpeza

**3. Moeda** — criei a coluna valor_brl com todos os valores na mesma unidade,
**preservando valor como veio da origem**. Rastreabilidade importa em PLD: se um
número for questionado, é preciso mostrar o valor original e o convertido lado a
lado

**4. Operação sem data (OP-001`)** — decidi manter a operação e excluí-la
apenas da Regra 1

Considerei dois caminhos: descartar a linha, mantê-la fora só da regra que depende
de data

Entre descartar e manter, o número decidiu: o cliente CLI-A-5 tem exatamente 4
operações, e OP-0017 é uma delas. A Regra 2 só se aplica a clientes com 4 ou
mais operações — descartar a linha reduziria o cliente a 3 e o tiraria inteiro do
escopo dessa regra. Um cliente deixaria de ser avaliado por uma regra que sequer
usa data. O defeito invalida a operação em um lugar só, então a exclusão deve
valer só nesse lugar



In [49]:
linhas_antes = len(df)

df = df.drop_duplicates()
df["data"] = pd.to_datetime(df["data"], errors="coerce")
df["valor_brl"] = df["valor"].astype(float)
df.loc[df["moeda"] == "USD", "valor_brl"] = df["valor"] * taxa_usd_brl
df = df.reset_index(drop=True)

print(f"Linhas antes:  {linhas_antes}")
print(f"Linhas depois: {len(df)}")
print(f"Sem data (fora da Regra 1): {df['data'].isna().sum()}")
print(f"Convertidas de USD: {(df['moeda'] == 'USD').sum()}")
print(f"\nTipo da coluna data: {df['data'].dtype}")

df[["id", "cliente_id", "data", "valor", "moeda", "valor_brl"]]


Linhas antes:  20
Linhas depois: 19
Sem data (fora da Regra 1): 1
Convertidas de USD: 1

Tipo da coluna data: datetime64[ns]


,id,cliente_id,data,valor,moeda,valor_brl
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,18100.0
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,17300.0
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,18800.0
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,3300.0
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,25900.0
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,27000.0
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,17200.0
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,15200.0
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,16100.0
9,OP-0010,CLI-A-4,2026-03-03,3800,BRL,3800.0
